In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoModel, AutoTokenizer, AutoConfig
from sklearn.metrics import roc_auc_score
import numpy as np
import pandas as pd
import logging
import random
import optuna
import functools
from collections import defaultdict
from tqdm.auto import tqdm
from typing import Dict, List, Optional
from sklearn.model_selection import StratifiedShuffleSplit
from collections import defaultdict
from sklearn.utils import resample
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score,StratifiedKFold
import json
from pathlib import Path

In [2]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [ ]:
PROJECT_ROOT = Path("../../").resolve()
DATA_DIR = PROJECT_ROOT/"data"

toxicity = "hepatotoxicity"
processed_assay_dir = DATA_DIR / "processed" / "bioassay" / toxicity / "assay_processed"
train_dir =  DATA_DIR / "processed" / "bioassay" / toxicity / "train"
test_dir = DATA_DIR / "processed" / "bioassay" / toxicity / "test"

train_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

In [3]:

logging.basicConfig(
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%H:%M:%S",
    level=logging.INFO
)
logger = logging.getLogger(__name__)

In [4]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [5]:
toxicity="hepatotoxicity"

In [ ]:
model_name =PROJECT_ROOT/"code/chembert_pfas/ChemBERTa-Full_FT-PFAS/best_full_finetuned_model" 
tokenizer = AutoTokenizer.from_pretrained(PROJECT_ROOT/"code/chembert_pfas/DeepChem/ChemBERTa-100M-MLM" )

In [ ]:
def resample_priority_pfas(df, target_n, label_value, random_state=42,is_priority_pfas=False):
    if is_priority_pfas :
        df_target = df[df["class"] == label_value]
        df_pfas = df_target[df_target["pfas_target"] == 1]
        df_normal = df_target[df_target["pfas_target"] == 0]
        
        n_pfas = len(df_pfas)
        if n_pfas >= target_n:
            return resample(df_pfas, replace=False, n_samples=target_n, random_state=random_state)
        else:
            selected_dfs = [df_pfas] 
            n_needed = target_n - n_pfas
            
            if n_needed > 0:
                if len(df_normal) >= n_needed:
                    df_normal_sampled = resample(df_normal, replace=False, n_samples=n_needed, random_state=random_state)
                    selected_dfs.append(df_normal_sampled)
                else:
                    selected_dfs.append(df_normal)
            
            return pd.concat(selected_dfs)
    else:
        df_target = df[df["class"] == label_value]
        return resample(df_target, replace=False, n_samples=target_n, random_state=random_state)

In [ ]:
def load_and_merge_datasets_final(dataset_configs, sampling_temp=0.5, min_retention=200, random_seed=42,is_priority_pfas=False):

    data_map = defaultdict(dict)
    task_names = set()
    
    pre_processed_dfs = {}
    step1_counts = {}

    logger.info(f"=== Starting data processing (PFAS priority + temperature balancing, alpha={sampling_temp}) ===")

    for config in dataset_configs:
        path = config['path']
        task = config['task_name']
        s_col = config['smiles_col']
        l_col = config['label_col']
        task_names.add(task)


        df = pd.read_csv(path)
        
        if "contains_CF" in df.columns:
            df = df[df["contains_CF"] == 1]
        df = df.dropna(subset=[s_col, l_col, "pfas_target"])
        
        n_pos = np.sum(df[l_col] == 1)
        n_neg = np.sum(df[l_col] == 0)
        
        if n_pos == 0 or n_neg == 0:
            logger.warning(f"Task [{task}] is missing a class (Pos={n_pos}, Neg={n_neg}); skipping")
            continue
            
        target_n_per_class = min(n_pos, n_neg)
        
        df_pos_bal = resample_priority_pfas(df, target_n_per_class, 1, random_state=random_seed, is_priority_pfas=is_priority_pfas)
        df_neg_bal = resample_priority_pfas(df, target_n_per_class, 0, random_state=random_seed, is_priority_pfas=is_priority_pfas)
        
        df_step1 = pd.concat([df_pos_bal, df_neg_bal])
        
        pre_processed_dfs[task] = df_step1
        step1_counts[task] = len(df_step1)
            
    if not step1_counts:
        raise ValueError("No valid data.")

    C = min_retention ** (1 - sampling_temp)
        
    final_target_counts = {}
    for task, n in step1_counts.items():
        raw_target = int(C * (n ** sampling_temp))
        
        target = min(n, raw_target)
        
        if target % 2 != 0: target -= 1
        final_target_counts[task] = max(2, target)

    for config in dataset_configs:
        task = config['task_name']
        if task not in pre_processed_dfs: continue
        
        df = pre_processed_dfs[task]
        current_n = step1_counts[task]
        target_n = final_target_counts[task]
        
        s_col = config['smiles_col']
        l_col = config['label_col']
        
        if target_n < current_n:
            df_final = resample(df, replace=False, n_samples=target_n, stratify=df[l_col], random_state=random_seed)
            logger.info(f"Task [{task}]: {current_n} -> {target_n} (reduced by {(1-target_n/current_n):.1%})")
        else:
            df_final = df
            logger.info(f"Task [{task}]: {current_n} -> {current_n} (retained)")
            
        for _, row in df_final.iterrows():
            smi = row[s_col]
            label = int(row[l_col])
            data_map[smi][task] = label

    all_smiles = list(data_map.keys())
    all_labels_list = list(data_map.values())
    task_names_list = sorted(list(task_names))
    
    logger.info(f"Processing complete. Total molecules: {len(all_smiles)}")
    return all_smiles, all_labels_list, task_names_list

In [ ]:
def split_raw_datasets(assay_ids, input_dir, output_train_dir, output_test_dir, test_size=0.2,random_seed=42):
    os.makedirs(output_train_dir, exist_ok=True)
    os.makedirs(output_test_dir, exist_ok=True)
    
    for aid in assay_ids:
        file_name = f"{aid}_processed.csv" 
        input_path = os.path.join(input_dir, file_name)
        
        if not os.path.exists(input_path):
            logger.warning(f"File does not exist: {input_path}")
            continue
            
        df = pd.read_csv(input_path)
        
        if "contains_CF" in df.columns:
            df = df[df["contains_CF"] == 1]
        
        df = df.dropna(subset=["standardized_smiles", "class", "pfas_target"])
        total_count = len(df)

        splitter = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_seed)
        train_idx, test_idx = next(splitter.split(df, df["class"]))
        
        train_df = df.iloc[train_idx]
        test_df = df.iloc[test_idx]

        train_save_path = os.path.join(output_train_dir, f"train_{aid}.csv")
        test_save_path = os.path.join(output_test_dir, f"test_{aid}.csv")
        
        train_df.to_csv(train_save_path, index=False)
        test_df.to_csv(test_save_path, index=False)
        
        logger.info(f"Task [{aid}]: Total={total_count} -> Train={len(train_df)}, Test={len(test_df)}")        


In [ ]:
data = pd.read_csv(
    PROJECT_ROOT / "code" / "bioassay_retrieval" / 
    f"{toxicity}_assay_desc_deduplicated_with_kappa_and_text_25.csv"
)
AIDS = data["assay_id"].to_list()

In [ ]:
split_raw_datasets(
    assay_ids=AIDS,
    input_dir=processed_assay_dir,
    output_train_dir=train_dir,
    output_test_dir=test_dir,
    test_size=0.2,
    random_seed=42
)

15:09:29 - INFO - Task [449763]: Total=43474 -> Train=34779, Test=8695
15:09:29 - INFO - Task [651597]: Total=97 -> Train=77, Test=20
15:09:29 - INFO - Task [743122]: Total=395 -> Train=316, Test=79
15:09:30 - INFO - Task [720637]: Total=311 -> Train=248, Test=63
15:09:30 - INFO - Task [743219]: Total=324 -> Train=259, Test=65
15:09:30 - INFO - Task [463112]: Total=116 -> Train=92, Test=24
15:09:30 - INFO - Task [1032]: Total=22359 -> Train=17887, Test=4472
15:09:31 - INFO - Task [504648]: Total=51130 -> Train=40904, Test=10226
15:09:31 - INFO - Task [504444]: Total=39152 -> Train=31321, Test=7831
15:09:32 - INFO - Task [2796]: Total=42359 -> Train=33887, Test=8472
15:09:32 - INFO - Task [2845]: Total=289 -> Train=231, Test=58
15:09:33 - INFO - Task [588692]: Total=47782 -> Train=38225, Test=9557
15:09:33 - INFO - Task [624491]: Total=216 -> Train=172, Test=44
15:09:33 - INFO - Task [1346985]: Total=149 -> Train=119, Test=30
15:09:33 - INFO - Task [1159563]: Total=143 -> Train=114, Tes

In [ ]:

train_files = [f for f in os.listdir(train_dir) if f.endswith(".csv")]

train_configs = []
for f in train_files:
    aid = f.split('_')[1].replace('.csv', '') 
    train_configs.append({
        "path": os.path.join(train_dir, f),
        "task_name": f"AID:{aid}",
        "smiles_col": "standardized_smiles",
        "label_col": "class"
    })

train_smiles, train_labels, task_names = load_and_merge_datasets_final(
    train_configs, 
    sampling_temp=0.5, 
    min_retention=256, 
    random_seed=42,
    is_priority_pfas=False 
)

logger.info(f"Training set ready: {len(train_smiles)} molecules")

15:09:33 - INFO - === 开始处理数据 (PFAS优先 + 温度平衡 alpha=0.5) ===
15:09:34 - INFO - --- 最终采样统计 ---
15:09:34 - INFO - Task [AID:1032]: 96 -> 96 (保持)
15:09:34 - INFO - Task [AID:1159563]: 82 -> 82 (保持)
15:09:34 - INFO - Task [AID:1347034]: 60 -> 60 (保持)
15:09:34 - INFO - Task [AID:1346985]: 50 -> 50 (保持)
15:09:34 - INFO - Task [AID:449763]: 204 -> 204 (保持)
15:09:34 - INFO - Task [AID:2796]: 1508 -> 620 (削减 58.9%)
15:09:34 - INFO - Task [AID:2845]: 56 -> 56 (保持)
15:09:34 - INFO - Task [AID:463112]: 66 -> 66 (保持)
15:09:34 - INFO - Task [AID:489024]: 40 -> 40 (保持)
15:09:34 - INFO - Task [AID:489027]: 52 -> 52 (保持)
15:09:34 - INFO - Task [AID:504444]: 1972 -> 710 (削减 64.0%)
15:09:34 - INFO - Task [AID:504648]: 122 -> 122 (保持)
15:09:34 - INFO - Task [AID:588692]: 156 -> 156 (保持)
15:09:34 - INFO - Task [AID:651597]: 68 -> 68 (保持)
15:09:34 - INFO - Task [AID:624491]: 82 -> 82 (保持)
15:09:34 - INFO - Task [AID:720589]: 62 -> 62 (保持)
15:09:34 - INFO - Task [AID:720637]: 116 -> 116 (保持)
15:09:34 - INFO - 

In [15]:
train_labels

[{'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1, 'AID:2796': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1, 'AID:2796': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 1},
 {'AID:1032': 0},
 {'AID:1032': 0},
 {'AID:1032': 0},
 {'AID:1032': 0},
 {'AID:1032': 0},
 {'AID:1032': 0}

In [ ]:
test_files = [f for f in os.listdir(test_dir) if f.endswith(".csv")]

test_configs = []
for f in test_files:
    aid = f.split('_')[1].replace('.csv', '')
    test_configs.append({
        "path": os.path.join(test_dir, f),
        "task_name": f"AID:{aid}",
        "smiles_col": "standardized_smiles",
        "label_col": "class"
    })
def load_test_datasets(configs):
    data_map = defaultdict(dict)
    for cfg in configs:
        df = pd.read_csv(cfg['path'])
        df = df.dropna(subset=[cfg['smiles_col'], cfg['label_col']])
        
        for _, row in df.iterrows():
            smi = row[cfg['smiles_col']]
            lbl = int(row[cfg['label_col']])
            data_map[smi][cfg['task_name']] = lbl
            
    return list(data_map.keys()), list(data_map.values())

test_smiles, test_labels = load_test_datasets(test_configs)

logger.info(f"Test set ready: {len(test_smiles)} molecules (original distribution retained)")

15:09:35 - INFO - 测试集准备完毕: 33963 个分子 (保持原始分布)


In [17]:
test_labels

[{'AID:1346985': 0, 'AID:504648': 0, 'AID:588692': 0},
 {'AID:1346985': 0},
 {'AID:1346985': 0, 'AID:743219': 0},
 {'AID:1346985': 0},
 {'AID:1346985': 0, 'AID:1032': 0},
 {'AID:1346985': 0, 'AID:1032': 0, 'AID:504648': 0},
 {'AID:1346985': 0},
 {'AID:1346985': 0},
 {'AID:1346985': 1, 'AID:2796': 0, 'AID:743122': 0},
 {'AID:1346985': 0, 'AID:743122': 0},
 {'AID:1346985': 1},
 {'AID:1346985': 0, 'AID:1347034': 0},
 {'AID:1346985': 0, 'AID:2796': 0, 'AID:743219': 0},
 {'AID:1346985': 1},
 {'AID:1346985': 0, 'AID:1347034': 0},
 {'AID:1346985': 0, 'AID:743122': 0, 'AID:743219': 0},
 {'AID:1346985': 0, 'AID:720637': 0},
 {'AID:1346985': 0, 'AID:743219': 1},
 {'AID:1346985': 0},
 {'AID:1346985': 1},
 {'AID:1346985': 0, 'AID:1347034': 0},
 {'AID:1346985': 0, 'AID:504648': 0, 'AID:720637': 0},
 {'AID:1346985': 0, 'AID:504444': 0},
 {'AID:1346985': 0},
 {'AID:1346985': 1},
 {'AID:1346985': 0,
  'AID:1032': 0,
  'AID:504648': 0,
  'AID:743122': 0,
  'AID:720637': 0},
 {'AID:1346985': 0},
 {'AID:

In [ ]:
class MoleculeDataset(torch.utils.data.Dataset):
    def __init__(self, smiles_list, labels_list, tokenizer, max_len=128):
        self.labels_list = labels_list
        self.encodings = tokenizer(smiles_list, truncation=True, padding=True, max_length=max_len)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels_dict'] = self.labels_list[idx]
        return item

    def __len__(self):
        return len(self.labels_list)

class MultitaskCollator:
    def __init__(self, task_names, ignore_index=-100):
        self.task_names = task_names
        self.task_to_idx = {name: i for i, name in enumerate(task_names)}
        self.ignore_index = ignore_index

    def __call__(self, batch):
        input_ids = torch.stack([item['input_ids'] for item in batch])
        attention_mask = torch.stack([item['attention_mask'] for item in batch])
        
        batch_size = len(batch)
        num_tasks = len(self.task_names)
        
        labels_tensor = torch.full((batch_size, num_tasks), self.ignore_index, dtype=torch.float)
        
        for i, item in enumerate(batch):
            sample_labels = item['labels_dict']
            for task_name, label_val in sample_labels.items():
                if task_name in self.task_to_idx:
                    col_idx = self.task_to_idx[task_name]
                    labels_tensor[i, col_idx] = label_val
        
        return input_ids, attention_mask, labels_tensor

In [19]:
train_dataset = MoleculeDataset(train_smiles, train_labels, tokenizer, max_len=128)
test_dataset = MoleculeDataset(test_smiles, test_labels, tokenizer, max_len=128)
train_collator = MultitaskCollator(task_names=task_names)
test_collator = MultitaskCollator(task_names=task_names)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,      
    shuffle=True,     
    collate_fn=train_collator, 
    num_workers=4,     
    pin_memory=True    
)
test_loader = DataLoader(
    test_dataset,
    batch_size=64,     
    shuffle=False,       
    collate_fn=test_collator,
    num_workers=4,
    pin_memory=True
)

In [ ]:
class MultiTaskChemBERTa(nn.Module):
    def __init__(self, model_name, task_names, head_dropout=0.1, head_hidden_dim=256):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.config = self.encoder.config
        self.task_names = task_names
        self.heads = nn.ModuleDict()

        self.task_log_vars = nn.ParameterDict()
        
        for name in task_names:
            self.heads[name] = nn.Sequential(
                nn.Dropout(head_dropout),
                nn.Linear(self.config.hidden_size, head_hidden_dim),
                nn.Tanh(),
                nn.Dropout(head_dropout),
                nn.Linear(head_hidden_dim, 1)
            )
            self.task_log_vars[name] = nn.Parameter(torch.zeros(()))

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :] 
        
        logits_dict = {}
        losses_dict = {} 
        
        for i, task_name in enumerate(self.task_names):
            logits = self.heads[task_name](pooled_output).squeeze(-1)
            logits_dict[task_name] = logits
            
            if labels is not None:
                task_labels = labels[:, i]
                mask = task_labels != -100
                
                if mask.sum() > 0:
                    valid_logits = logits[mask]
                    valid_labels = task_labels[mask]
                    
                    loss_fct = nn.BCEWithLogitsLoss() 
                    raw_loss = loss_fct(valid_logits, valid_labels)
                
                    precision = torch.exp(-self.task_log_vars[task_name])
                    weighted_loss = 0.5 * precision * raw_loss + 0.5 * self.task_log_vars[task_name]
                    
                    losses_dict[task_name] = weighted_loss
                    
        return {"logits": logits_dict, "losses_dict": losses_dict}
    
    
import os
import json
import torch

def save_huggingface_model(model, tokenizer, output_dir, pruned_tasks=None):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)

    print(f"Saving model to {output_dir}...")

    tokenizer.save_pretrained(output_dir)

    config_dict = model.config.to_dict()

    config_dict.update({
        "architectures": ["MultiTaskChemBERTa"], 
        "task_names": model.task_names,          
        "head_hidden_dim": model.heads[model.task_names[0]][1].out_features, 
        "head_dropout": model.heads[model.task_names[0]][0].p,
        "pruned_tasks": pruned_tasks if pruned_tasks is not None else []
    })
    with open(os.path.join(output_dir, "config.json"), "w", encoding="utf-8") as f:
        json.dump(config_dict, f, indent=2, ensure_ascii=False)

    torch.save(model.state_dict(), os.path.join(output_dir, "pytorch_model.bin"))
    
    print("model saved successfully!")
    if pruned_tasks:
        print(f"Recorded {len(pruned_tasks)} pruned tasks in the configuration file.")
    
def load_huggingface_model(model_dir, device="cpu"):
    print(f"Loading model from {model_dir}...")
    config = AutoConfig.from_pretrained(model_dir)
    
    task_names = getattr(config, "task_names", [])
    head_hidden_dim = getattr(config, "head_hidden_dim", 512)
    head_dropout = getattr(config, "head_dropout", 0.1)
    
    pruned_tasks = getattr(config, "pruned_tasks", [])
    
    if len(pruned_tasks) > 0:
        print(f"   {pruned_tasks}")
    
    model = MultiTaskChemBERTa(
        model_name=model_dir, 
        task_names=task_names,
        head_hidden_dim=head_hidden_dim,
        head_dropout=head_dropout
    )
    
    state_dict = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    
    model.to(device)
    model.eval()
    
    model.pruned_tasks = pruned_tasks
    
    return model, tokenizer

In [23]:
optuna.logging.set_verbosity(optuna.logging.INFO)

In [ ]:
class SmartTrainer:
    def __init__(self, model, train_loader, val_loader, device, task_names, 
                 pruning_start_epoch=3, pruning_threshold=0.55):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.task_names = task_names
        self.optimizer = None 
        self.pruning_start_epoch = pruning_start_epoch
        self.pruning_threshold = pruning_threshold
        self.active_tasks = {name: True for name in task_names}
        self.pruned_tasks = []

    def set_freeze_encoder(self, freeze: bool):
        for param in self.model.encoder.parameters():
            param.requires_grad = not freeze
            
    def train_epoch(self):
        self.model.train()
        total_loss_val = 0
        steps = 0
        for batch in self.train_loader:
            input_ids, attention_mask, labels = [b.to(self.device) for b in batch]
            self.optimizer.zero_grad()
            output = self.model(input_ids, attention_mask, labels=labels)
        
            batch_loss = torch.tensor(0.0, device=self.device)
            valid_tasks_count = 0
            
            for task_name, task_loss in output['losses_dict'].items():
                if self.active_tasks[task_name]:
                    batch_loss += task_loss
                    valid_tasks_count += 1
            if valid_tasks_count == 0 or batch_loss.item() == 0.0:
                continue

            batch_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optimizer.step()
            total_loss_val += batch_loss.item()
            steps += 1
            
        return total_loss_val / max(steps, 1)

    @torch.no_grad()
    def evaluate(self):
        self.model.eval()
        all_preds = {t: [] for t in self.task_names}
        all_labels = {t: [] for t in self.task_names}
        
        for batch in self.val_loader:
            input_ids, attention_mask, labels = [b.to(self.device) for b in batch]
            output = self.model(input_ids, attention_mask)
            
            for i, task in enumerate(self.task_names):
                preds = torch.sigmoid(output['logits'][task]).cpu().numpy()
                lbls = labels[:, i].cpu().numpy()
                mask = lbls != -100
                if mask.any():
                    all_preds[task].extend(preds[mask])
                    all_labels[task].extend(lbls[mask])
        
        scores_task = {}
        valid_scores = []
        
        for task in self.task_names:
            y_true = np.array(all_labels[task])
            y_score = np.array(all_preds[task])
            
            if len(np.unique(y_true)) > 1:
                try:
                    s = roc_auc_score(y_true, y_score)
                    scores_task[task] = s
                    if self.active_tasks[task]:
                        valid_scores.append(s)
                except ValueError:
                    scores_task[task] = np.nan
            else:
                scores_task[task] = np.nan
                
        return np.mean(valid_scores) if valid_scores else 0.0, scores_task

    def check_and_prune(self, epoch, scores_task):

        if epoch < self.pruning_start_epoch:
            return

        for task, auc in scores_task.items():
            if self.active_tasks[task]:
                if np.isnan(auc) or auc < self.pruning_threshold:
                    print(f"Task {task} (AUC={auc:.4f}) was pruned; training for this task has stopped.")
                    self.active_tasks[task] = False
                    self.pruned_tasks.append(task)

In [25]:
set_seed(42)
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")

In [ ]:

p_batch_size = 32
p_dropout = 0.4
p_hidden_dim = 512
p_lr_stage1 = 0.0004
p_lr_stage2 = 1e-5 


collate_fn = MultitaskCollator(task_names)
train_loader = DataLoader(train_dataset, batch_size=p_batch_size, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(test_dataset, batch_size=p_batch_size, shuffle=False, collate_fn=collate_fn, num_workers=0)


model = MultiTaskChemBERTa(
    model_name, 
    task_names, 
    head_dropout=p_dropout, 
    head_hidden_dim=p_hidden_dim
).to(device)


trainer = SmartTrainer(
    model, train_loader, val_loader, device, task_names,
    pruning_start_epoch=3,  
    pruning_threshold=0.55  
)

print("--- Stage 1: Encoder Frozen Warmup ---")
trainer.set_freeze_encoder(True)
trainer.optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=p_lr_stage1)
trainer.train_epoch()
    
print("--- Stage 2: Full Fine-tuning with Dynamic Pruning ---")
trainer.set_freeze_encoder(False)
trainer.optimizer = AdamW(model.parameters(), lr=p_lr_stage2)

best_val_auc = -np.inf  
epochs = 20
early_stop_counter = 0  
p_patience=5             
best_model_path = "best_model_checkpoint.pth" 

for epoch in range(epochs):
    loss = trainer.train_epoch()
    avg_auc, scores_task = trainer.evaluate()
    
    trainer.check_and_prune(epoch, scores_task)
    
    print(f"Epoch {epoch+1}/20 - Loss: {loss:.4f} - Avg Active AUC: {avg_auc:.4f}")
    print(f"active tasks nums{len([t for t, active in trainer.active_tasks.items() if active])}, total tasks nums {len(trainer.active_tasks)}")
    for task, score in scores_task.items():
        try:
            aid = int(task.split(':')[1])
            data.loc[data['assay_id'] == aid, f'epoch{epoch+1}_auc'] = score
            status = "ACTIVE" if trainer.active_tasks[task] else "PRUNED"
            print(f"  {task}: {score:.4f} [{status}]")
        except:
            pass
    if avg_auc > best_val_auc:
        print(f"  Performance improved ({best_val_auc:.4f} -> {avg_auc:.4f}). Saving the best model...")
        best_val_auc = avg_auc
        early_stop_counter = 0

        save_huggingface_model(
            model, 
            tokenizer, 
            output_dir="best_model_with_pruning_heptox", 
            pruned_tasks=trainer.pruned_tasks
        )
    else:
        early_stop_counter += 1
        print(f"  Starting Early Stopping counter: {early_stop_counter} / {p_patience}")
        
        if early_stop_counter >= p_patience:
            print(f"\nEarly stopping triggered: no performance improvement for {p_patience} consecutive epochs.")
            print(f"   Best AUC: {best_val_auc:.4f}")
            break 


Some weights of RobertaModel were not initialized from the model checkpoint at ../微调chemberta/ChemBERTa-Full_FT-PFAS-1-11/best_full_finetuned_model and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


--- Stage 1: Encoder Frozen Warmup ---
--- Stage 2: Full Fine-tuning with Dynamic Pruning ---
Epoch 1/20 - Loss: 4.1307 - Avg Active AUC: 0.6905
  AID:1032: 0.6013 [ACTIVE]
  AID:1159563: 0.7980 [ACTIVE]
  AID:1346985: 0.7986 [ACTIVE]
  AID:1347034: 0.6750 [ACTIVE]
  AID:2796: 0.8048 [ACTIVE]
  AID:2845: 0.4006 [ACTIVE]
  AID:449763: 0.8168 [ACTIVE]
  AID:463112: 0.6250 [ACTIVE]
  AID:489024: 0.4600 [ACTIVE]
  AID:489027: 0.6667 [ACTIVE]
  AID:504444: 0.7361 [ACTIVE]
  AID:504648: 0.8431 [ACTIVE]
  AID:588692: 0.8199 [ACTIVE]
  AID:624491: 0.4853 [ACTIVE]
  AID:651597: 0.6566 [ACTIVE]
  AID:720589: 0.5268 [ACTIVE]
  AID:720637: 0.9111 [ACTIVE]
  AID:743122: 0.7460 [ACTIVE]
  AID:743219: 0.6513 [ACTIVE]
  AID:743395: 0.7785 [ACTIVE]
  AID:743416: 0.7000 [ACTIVE]
  ★ 性能提升 (-inf -> 0.6905)! 保存最佳模型...
正在保存模型至 best_model_with_pruning ...
model saved successfully!
Epoch 2/20 - Loss: 3.5760 - Avg Active AUC: 0.7053
  AID:1032: 0.6323 [ACTIVE]
  AID:1159563: 0.7980 [ACTIVE]
  AID:1346985: 0.79